In [ ]:
import numpy as np
import pandas as pd
from pulp import *

In [ ]:
def generate_mock_duty_matrix(m=100, n=500, density=0.05):
    # 生成一個隨機的 0/1 矩陣，模擬 m 個 PoW 與 n 個 Duty
    # density 代表平均一個 Duty 包含多少比例的 PoW
    matrix = np.random.choice([0, 1], size=(m, n), p=[1-density, density])
    
    # 確保每一行至少被一個 Duty 覆蓋 (避免無解)
    for i in range(m):
        if np.sum(matrix[i, :]) == 0:
            matrix[i, np.random.randint(0, n)] = 1
            
    return matrix

In [ ]:
# 測試用參數
m_test, n_test = 100, 500
matrix_A = generate_mock_duty_matrix(m_test, n_test)

In [ ]:
# 轉成 DataFrame，設定好欄位名稱方便閱讀
df_A = pd.DataFrame(matrix_A)
df_A.columns = [f'Duty_{j}' for j in range(n_test)]
df_A.index = [f'PoW_{i}' for i in range(m_test)]

# 儲存成 CSV
df_A.to_csv('mock_duty_matrix_100_500.csv')
print("測試矩陣已存入 mock_duty_matrix_100_500.csv")

In [ ]:
def load_matrix_from_csv(file_name):
    # 1. 讀取 CSV，指定第一列為 Index (PoW_i)
    df = pd.read_csv(file_name, index_col=0)
    
    # 2. 提取數值部分 (values 會回傳 NumPy array)
    matrix_A = df.values
    
    # 3. 獲取維度資訊，確保跟後續變數對得起來
    m, n = matrix_A.shape
    
    print(f"成功讀取矩陣：共 {m} 個 PoW，{n} 個 Duty")
    return matrix_A, m, n

# 實作調用
matrix_A, m_test, n_test = load_matrix_from_csv('mock_duty_matrix_100_500.csv')

In [ ]:
# 建立問題
prob = LpProblem("Driver_Scheduling_Shortage_Test", LpMinimize)

# 參數設定
alpha = 1.0     # 司機權重
beta = 2.0      # 懲罰權重
N = 10          # 人力天花板

# 變數定義
x = LpVariable.dicts("Duty", range(n_test), cat='Binary')
P = LpVariable.dicts("Penalty", range(m_test), cat='Binary')
C = {i: (100 if i < 30 else 1) for i in range(m_test)} # 假設前20個是核心路線 (Ci=100)

# 目標函數：min (alpha * sum x_j + beta * sum Pi * Ci)
prob += alpha * lpSum([x[j] for j in range(n_test)]) + \
        beta * lpSum([P[i] * C[i] for i in range(m_test)])

# 約束條件 1: 覆蓋或放棄
# 預處理：先找出每個 PoW i 被哪些 Duty j 覆蓋
pow_to_duties = {i: [] for i in range(m_test)}
for j in range(n_test):
    for i in np.where(matrix_A[:, j] == 1)[0]: # 假設 A 是 numpy array
        pow_to_duties[i].append(j)

# 在 PuLP 約束中直接使用
for i in range(m_test):
    # 直接從預處理好的清單抓變數，速度快很多
    covered_by = [x[j] for j in pow_to_duties[i]]
    prob += lpSum(covered_by) + P[i] >= 1

# 約束條件 2: 人力天花板 (關鍵測試點)
prob += lpSum([x[j] for j in range(n_test)]) <= N

# 求解
prob.solve()

In [ ]:
for v in prob.variables():
    if v.varValue == 1:
        print(v.name, "=", v.varValue)

In [ ]:
if LpStatus[prob.status] == 'Optimal':
    print(f"Total Cost: {value(prob.objective)}")
    
    # 分別計算兩項的貢獻
    drivers_used = sum(value(x[j]) for j in range(n_test))
    total_penalty = sum(value(P[i]) * C[i] for i in range(m_test))
    
    print(f"Number of Drivers: {drivers_used}")
    print(f"Total Penalty: {total_penalty}")

In [ ]:
# 設定輸出檔案路徑
output_file = "demo_result_1.txt"

with open(output_file, "w", encoding="utf-8") as f:
    # 1. 寫入實驗參數設定
    f.write("===== 實驗參數設定 (Demo) =====\n")
    f.write(f"司機權重 (alpha): {alpha}\n")
    f.write(f"懲罰權重 (beta): {beta}\n")
    f.write(f"人力天花板 (N): {N}\n\n")
    
    # 2. 寫入成本 (Cost) 定義邏輯
    f.write("===== 成本定義邏輯 =====\n")
    f.write("核心路線 (i < 20): Ci = 100\n")
    f.write("普通路線 (i >= 20): Ci = 1\n\n")
    
    # 3. 寫入求解結果
    f.write("===== 求解結果 =====\n")
    f.write(f"求解狀態: {LpStatus[prob.status]}\n")
    
    if LpStatus[prob.status] == 'Optimal':
        f.write(f"總成本 (Total Cost): {value(prob.objective)}\n")
        
        drivers_used = sum(value(x[j]) for j in range(n_test))
        total_penalty = sum(value(P[i]) * C[i] for i in range(m_test))
        
        f.write(f"使用的司機人數: {drivers_used}\n")
        f.write(f"總懲罰分數: {total_penalty}\n\n")
        
        # 4. 寫入具體的決策變數 (選了哪些 Duty / 哪些班次被放棄)
        f.write("===== 具體決策細節 =====\n")
        f.write("[指派的 Duty]:\n")
        selected_duties = [v.name for v in prob.variables() if v.name.startswith("Duty") and v.varValue == 1]
        f.write(f"{', '.join(selected_duties) if selected_duties else '無'}\n\n")
        
        f.write("[放棄的班次 (P_i = 1)]:\n")
        cancelled_trips = [v.name for v in prob.variables() if v.name.startswith("Penalty") and v.varValue == 1]
        f.write(f"{', '.join(cancelled_trips) if cancelled_trips else '無'}\n")
    else:
        f.write("未找到最優解，請檢查約束條件或 N 值是否過小。\n")

print(f"Demo 結果已成功寫入 {output_file}")